In [2]:
# lowkey just copy what i did in exposed_mutations but get the middle + last nucleotide
#i.e. get mutational contexts for SNPs -> dinucleotide context

In [3]:
import pandas as pd
import pysam
import os

In [4]:
# load nucleosome file
nuc_path = "/home/ubuntu/honors_research/peak_align/reference_files/edited_TSS_all_adj+1Nuc_1.bed"
nuc_df = pd.read_csv(nuc_path, sep="\t", header=None, names=["chrom", "start", "end", "name", "score", "strand", "distance", "phase"])


In [5]:
nuc_df

,chrom,start,end,name,score,strand,distance,phase
0,chr1,21782419,21782419,chr1_21782419_21782419,5,+,-19,-3_AA_-2_-1_TA_0_1_CT_2_c_main_NDR_First_Diver...
1,chr16,68085718,68085718,chr16_68085718_68085718,16,+,-8,-3_CT_-2_-1_CG_0_1_AC_2_c_main_NDR_First_Diver...
2,chr1,148634329,148634329,chr1_148634329_148634329,5,-,40,-3_AT_-2_-1_TA_0_1_CT_2_c_main_NFR_First_Singl...
3,chr1,156747365,156747365,chr1_156747365_156747365,7,+,43,-3_AC_-2_-1_CA_0_1_CA_2_c_main_NFR_First_Diver...
4,chr1,172449989,172449989,chr1_172449989_172449989,10,-,40,-3_TC_-2_-1_CA_0_1_CT_2_c_main_NFR_First_Refer...
...,...,...,...,...,...,...,...,...
51311,chr18,57435408,57435408,chr18_57435408_57435408,126,+,967,-3_AA_-2_-1_CA_0_1_GA_2_c_main_NFR_First_Refer...
51312,chr19,37251877,37251877,chr19_37251877_37251877,5,+,988,-3_CC_-2_-1_CA_0_1_CG_2_c_main_NFR_First_Singl...
51313,chr8,1701428,1701428,chr8_1701428_1701428,7,-,1000,-3_CG_-2_-1_AG_0_1_GG_2_c_main_NFR_First_Diver...
51314,chr13,108215535,108215535,chr13_108215535_108215535,9,-,21,-3_GG_-2_-1_CA_0_1_TC_2_c_main_NDR_First_Diver...


In [6]:
def add_context_column(df, reference_genome_path):
    """
    Adds a 'context' column to the DataFrame with the nucleotide context (before, SNP, after).

    Parameters:
    df (pd.DataFrame): DataFrame containing mutation information with 'chrom', 'start', and 'end' columns (0-based).
    reference_genome_path (str): Path to the reference genome (e.g., hg38.fa.gz).

    Returns:
    pd.DataFrame: Updated DataFrame with a new 'context' column.
    """
    # Load the reference genome
    fasta = pysam.FastaFile(reference_genome_path)

    # Function to fetch context for a single mutation
    def fetch_context(row):
        try:
            chrom = row['chrom']
            pos = row['start'] +1  # Convert 0-based to 1-based for pysam
            
            before_2 = fasta.fetch(chrom, pos - 3, pos - 2) # Nucleotide before -1
            before = fasta.fetch(chrom, pos - 2, pos - 1)  # Nucleotide before
            ref = fasta.fetch(chrom, pos - 1, pos)  # Reference nucleotide at the SNP position
            after = fasta.fetch(chrom, pos, pos + 1)  # Nucleotide after
            after_2 = fasta.fetch(chrom, pos + 1, pos + 2) # Nucleotide after +1

            # Making everything in caps
            before = before.upper()
            before_2 = before_2.upper()
            ref = ref.upper()
            after = after.upper()
            after_2 = after_2.upper()



            #return f"{before}{ref}{after}"
            return f"{before_2}{before}{ref}{after}{after_2}"
        except KeyError:
            # Handle cases where the chromosome is not found in the reference genome
            return "Context not found"

    # Apply the fetch_context function to each row in the DataFrame
    df['context'] = df.apply(fetch_context, axis=1)

    return df


In [7]:
dinucleotide = add_context_column(nuc_df, "/home/ubuntu/honors_research/hg38.fa")

In [8]:
dinucleotide

,chrom,start,end,name,score,strand,distance,phase,context
0,chr1,21782419,21782419,chr1_21782419_21782419,5,+,-19,-3_AA_-2_-1_TA_0_1_CT_2_c_main_NDR_First_Diver...,TACTA
1,chr16,68085718,68085718,chr16_68085718_68085718,16,+,-8,-3_CT_-2_-1_CG_0_1_AC_2_c_main_NDR_First_Diver...,CGACT
2,chr1,148634329,148634329,chr1_148634329_148634329,5,-,40,-3_AT_-2_-1_TA_0_1_CT_2_c_main_NFR_First_Singl...,AGTAA
3,chr1,156747365,156747365,chr1_156747365_156747365,7,+,43,-3_AC_-2_-1_CA_0_1_CA_2_c_main_NFR_First_Diver...,CACAG
4,chr1,172449989,172449989,chr1_172449989_172449989,10,-,40,-3_TC_-2_-1_CA_0_1_CT_2_c_main_NFR_First_Refer...,AGTGG
...,...,...,...,...,...,...,...,...,...
51311,chr18,57435408,57435408,chr18_57435408_57435408,126,+,967,-3_AA_-2_-1_CA_0_1_GA_2_c_main_NFR_First_Refer...,CAGAA
51312,chr19,37251877,37251877,chr19_37251877_37251877,5,+,988,-3_CC_-2_-1_CA_0_1_CG_2_c_main_NFR_First_Singl...,CACGT
51313,chr8,1701428,1701428,chr8_1701428_1701428,7,-,1000,-3_CG_-2_-1_AG_0_1_GG_2_c_main_NFR_First_Diver...,CCCTC
51314,chr13,108215535,108215535,chr13_108215535_108215535,9,-,21,-3_GG_-2_-1_CA_0_1_TC_2_c_main_NDR_First_Diver...,GATGC


In [9]:
#dinucleotide recoding
def dinucleotide_recoding(context_column):
            

    before_2 = context_column[0]
    before = context_column[1]
    ref = context_column[2]
    after = context_column[3]
    after_2 = context_column[4]
    
    before = 'W' if before in ['A', 'T'] else 'S'
    before_2 = 'W' if before_2 in ['A', 'T'] else 'S'
    ref = 'W' if ref in ['A', 'T'] else 'S'
    after = 'W' if after in ['A', 'T'] else 'S'
    after_2 = 'N'

    return f"{before_2}{before}{ref}{after}{after_2}"

In [10]:
# Apply the recoding function to the 'context' column, and create a new column for results
dinucleotide['recoded_context'] = dinucleotide['context'].apply(dinucleotide_recoding)

In [11]:
dinucleotide

,chrom,start,end,name,score,strand,distance,phase,context,recoded_context
0,chr1,21782419,21782419,chr1_21782419_21782419,5,+,-19,-3_AA_-2_-1_TA_0_1_CT_2_c_main_NDR_First_Diver...,TACTA,WWSWN
1,chr16,68085718,68085718,chr16_68085718_68085718,16,+,-8,-3_CT_-2_-1_CG_0_1_AC_2_c_main_NDR_First_Diver...,CGACT,SSWSN
2,chr1,148634329,148634329,chr1_148634329_148634329,5,-,40,-3_AT_-2_-1_TA_0_1_CT_2_c_main_NFR_First_Singl...,AGTAA,WSWWN
3,chr1,156747365,156747365,chr1_156747365_156747365,7,+,43,-3_AC_-2_-1_CA_0_1_CA_2_c_main_NFR_First_Diver...,CACAG,SWSWN
4,chr1,172449989,172449989,chr1_172449989_172449989,10,-,40,-3_TC_-2_-1_CA_0_1_CT_2_c_main_NFR_First_Refer...,AGTGG,WSWSN
...,...,...,...,...,...,...,...,...,...,...
51311,chr18,57435408,57435408,chr18_57435408_57435408,126,+,967,-3_AA_-2_-1_CA_0_1_GA_2_c_main_NFR_First_Refer...,CAGAA,SWSWN
51312,chr19,37251877,37251877,chr19_37251877_37251877,5,+,988,-3_CC_-2_-1_CA_0_1_CG_2_c_main_NFR_First_Singl...,CACGT,SWSSN
51313,chr8,1701428,1701428,chr8_1701428_1701428,7,-,1000,-3_CG_-2_-1_AG_0_1_GG_2_c_main_NFR_First_Diver...,CCCTC,SSSWN
51314,chr13,108215535,108215535,chr13_108215535_108215535,9,-,21,-3_GG_-2_-1_CA_0_1_TC_2_c_main_NDR_First_Diver...,GATGC,SWWSN


In [12]:
# table for dinucleotide contexts counts
dinucleotide['recoded_context'].value_counts().reset_index()



,recoded_context,count
0,SWSWN,9764
1,WSWSN,9177
2,SWSSN,3919
3,SSWSN,3819
4,SWWSN,3756
5,SWWWN,3531
6,WWWSN,3349
7,WWSWN,2543
8,WSWWN,2463
9,WWWWN,1798


In [19]:
#high S = >= 4 S
#medium S = 2-3 S
#low = 1


def categorize_dinucleotide(context):
    s_count = context.count("S")  # how many G/C positions
    if s_count >= 4:
        return "High_S"
    elif 2<= s_count <= 3:
        return "Medium_S"
    elif s_count <= 1:
        return "Low_S"
 

print(dinucleotide['context'].head())


0    TACTA
1    CGACT
2    AGTAA
3    CACAG
4    AGTGG
Name: context, dtype: object


In [21]:
dinucleotide['category'] = dinucleotide['recoded_context'].apply(categorize_dinucleotide)

In [22]:
dinucleotide

,chrom,start,end,name,score,strand,distance,phase,context,recoded_context,category,s_count
0,chr1,21782419,21782419,chr1_21782419_21782419,5,+,-19,-3_AA_-2_-1_TA_0_1_CT_2_c_main_NDR_First_Diver...,TACTA,WWSWN,Low_S,0
1,chr16,68085718,68085718,chr16_68085718_68085718,16,+,-8,-3_CT_-2_-1_CG_0_1_AC_2_c_main_NDR_First_Diver...,CGACT,SSWSN,Medium_S,0
2,chr1,148634329,148634329,chr1_148634329_148634329,5,-,40,-3_AT_-2_-1_TA_0_1_CT_2_c_main_NFR_First_Singl...,AGTAA,WSWWN,Low_S,0
3,chr1,156747365,156747365,chr1_156747365_156747365,7,+,43,-3_AC_-2_-1_CA_0_1_CA_2_c_main_NFR_First_Diver...,CACAG,SWSWN,Medium_S,0
4,chr1,172449989,172449989,chr1_172449989_172449989,10,-,40,-3_TC_-2_-1_CA_0_1_CT_2_c_main_NFR_First_Refer...,AGTGG,WSWSN,Medium_S,0
...,...,...,...,...,...,...,...,...,...,...,...,...
51311,chr18,57435408,57435408,chr18_57435408_57435408,126,+,967,-3_AA_-2_-1_CA_0_1_GA_2_c_main_NFR_First_Refer...,CAGAA,SWSWN,Medium_S,0
51312,chr19,37251877,37251877,chr19_37251877_37251877,5,+,988,-3_CC_-2_-1_CA_0_1_CG_2_c_main_NFR_First_Singl...,CACGT,SWSSN,Medium_S,0
51313,chr8,1701428,1701428,chr8_1701428_1701428,7,-,1000,-3_CG_-2_-1_AG_0_1_GG_2_c_main_NFR_First_Diver...,CCCTC,SSSWN,Medium_S,0
51314,chr13,108215535,108215535,chr13_108215535_108215535,9,-,21,-3_GG_-2_-1_CA_0_1_TC_2_c_main_NDR_First_Diver...,GATGC,SWWSN,Medium_S,0


In [27]:
# separate into separate data frames

strong_df = dinucleotide[dinucleotide['category'] == 'High_S']
weak_df = dinucleotide[dinucleotide['category'] == 'Low_S']
med_df = dinucleotide[dinucleotide['category'] == 'Medium_S']

In [28]:
weak_df

,chrom,start,end,name,score,strand,distance,phase,context,recoded_context,category,s_count
0,chr1,21782419,21782419,chr1_21782419_21782419,5,+,-19,-3_AA_-2_-1_TA_0_1_CT_2_c_main_NDR_First_Diver...,TACTA,WWSWN,Low_S,0
2,chr1,148634329,148634329,chr1_148634329_148634329,5,-,40,-3_AT_-2_-1_TA_0_1_CT_2_c_main_NFR_First_Singl...,AGTAA,WSWWN,Low_S,0
8,chr1,33080995,33080995,chr1_33080995_33080995,57,-,42,-3_AC_-2_-1_TG_0_1_TA_2_c_main_NFR_First_Refer...,TACAG,WWSWN,Low_S,0
13,chr1,8005288,8005288,chr1_8005288_8005288,4,-,41,-3_TT_-2_-1_CA_0_1_TT_2_c_main_NFR_First_Diver...,AATGA,WWWSN,Low_S,0
15,chr10,50362885,50362885,chr10_50362885_50362885,5,-,43,-3_CC_-2_-1_TG_0_1_AA_2_c_main_NFR_First_Singl...,TTCAG,WWSWN,Low_S,0
...,...,...,...,...,...,...,...,...,...,...,...,...
51277,chr19,34254568,34254568,chr19_34254568_34254568,47,+,580,-3_TG_-2_-1_TA_0_1_CA_2_c_main_NFR_First_Diver...,TACAC,WWSWN,Low_S,0
51279,chr22,49827870,49827870,chr22_49827870_49827870,179,-,582,-3_CG_-2_-1_CA_0_1_TT_2_c_main_NFR_First_Refer...,AATGC,WWWSN,Low_S,0
51295,chr19,797452,797452,chr19_797452_797452,722,+,39,-3_GC_-2_-1_TA_0_1_TT_2_c_main_NDR_First_Refer...,TATTC,WWWWN,Low_S,0
51301,chr19,54697162,54697162,chr19_54697162_54697162,6,+,787,-3_GA_-2_-1_TG_0_1_AT_2_c_main_NFR_First_Refer...,TGATG,WSWWN,Low_S,0


In [29]:
# exporting (dropping the category column)

strong_df = strong_df.drop(columns=['context', 'category', 'recoded_context', 's_count'])
weak_df = weak_df.drop(columns=['context', 'category', 'recoded_context', 's_count'])
med_df = med_df.drop(columns=['context', 'category', 'recoded_context', 's_count'])



In [31]:
strong_df.to_csv('/home/ubuntu/honors_research/peak_align/dinucleotide_analysis/strong/strong_dinucleotide_nuc.bed', sep='\t', index=False, header=False)
weak_df.to_csv('/home/ubuntu/honors_research/peak_align/dinucleotide_analysis/weak/weak_dinucleotide_nuc.bed', sep='\t', index=False, header=False)
med_df.to_csv('/home/ubuntu/honors_research/peak_align/dinucleotide_analysis/med/med_dinucleotide_nuc.bed', sep='\t', index=False, header=False)
